# Lab 4 — Chat With YOUR Notes
**Session 4 · Embeddings + RAG · TCE** — the weekend's main build (~60 lines, all yours)

You need: your 2–3 documents (PDF or .txt). Upload via folder icon. **File → Save a copy in Drive** first.

No document, or a scanned PDF? Cell 2 falls back to the instructor sample `sample-os-notes.txt` (OS lecture notes + an *illustrative* regulations page) and Cells 5–7 are pre-filled with queries that work on it — swap in your own as soon as your search works.

In [ ]:
# Cell 1 — setup
%pip install -q -U google-genai pypdf numpy
from getpass import getpass
from google import genai
from google.genai import types
import numpy as np, time, re, os, json, zlib

MOCK = False   # ← set True only if the instructor says the live API is unavailable

client = genai.Client(api_key="mock" if MOCK else getpass("Gemini API key: "))
MODEL = "gemini-flash-lite-latest"  # lite: minimal thinking by default (near-zero thinking tokens on simple prompts), so the free tier goes further (Aug 2026 → Gemini 3.5 Flash Lite). 429 = rate limit: the helper below waits and retries; 503 = high demand: wait and re-run.
EMBED_MODEL = "gemini-embedding-2"   # check https://ai.google.dev/gemini-api/docs/embeddings

# Capstone insurance: Colab forgets its disk at lunch, Drive does not.
# Cell 4 saves your embeddings here; Labs 5 and 6 reload them from the same folder.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/genai'
except Exception as e:                      # not on Colab, or you declined the mount → local runtime disk
    print("Drive not mounted (" + type(e).__name__ + ") — using the local runtime disk instead")
    SAVE_DIR = '.'
os.makedirs(SAVE_DIR, exist_ok=True)
print("saving to", SAVE_DIR)

_MOCK_ANSWERS = [   # (keyword in the QUESTION, canned reply) — offline path only, not real model output
    ("rank these passages", '{"ids": [0, 1, 2, 3]}'),
    ("attendance",  "A minimum of 75% attendance is required to write the end-semester exam; 65–75% may be condoned on medical grounds [1]."),
    ("pass",        "Two conditions: at least 45% in the end-semester exam AND 50% of the total marks in the course [1]."),
    ("deadlock",    "Mutual exclusion, hold and wait, no preemption and circular wait — all four must hold at once [1]."),
    ("thread",      "Threads of one process share code, data and heap but each has its own stack and registers; processes are isolated from each other [1]."),
    ("quantum",     "Each process runs for one time quantum, then goes to the back of the ready queue; a very large quantum degenerates to FCFS [2]."),
    ("belady",      "FIFO suffers Belady's anomaly — more frames can mean MORE page faults; LRU does not [1]."),
    ("paging",      "Paging: fixed-size pages, invisible to the programmer. Segmentation: variable-size, mirrors the program's structure [1]."),
    ("internal",    "Two tests of 20 marks scaled to 30, plus 20 marks for assignments and quizzes — 50 marks internal in total [1]."),
    ("arrear",      "An arrear means re-appearing for the end-semester exam only; internals are retained; more than 4 standing arrears blocks higher-semester registration [1] [2]."),
]
_MOCK_DEFAULT = "I don't know based on the provided documents."

def _mock(prompt):
    p = str(prompt).lower()
    if "rank these passages" in p:
        return "[MOCK] " + _MOCK_ANSWERS[0][1]
    q = p.split("question:")[-1]             # match on the question, not on the retrieved context
    return "[MOCK] " + next((a for k, a in _MOCK_ANSWERS if k in q), _MOCK_DEFAULT)

def ask(contents, temperature=0.0):
    if MOCK:
        return _mock(contents)
    for attempt in range(4):
        try:
            r = client.models.generate_content(
                model=MODEL, contents=contents,
                config=types.GenerateContentConfig(temperature=temperature))
            return r.text or "[empty/blocked response]"
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print("rate limited..."); time.sleep(20*(attempt+1))
            else: raise
print("ready ✓" + ("  (MOCK mode — canned answers, nothing leaves this notebook)" if MOCK else ""))

## Part A — Ingest: load → chunk → embed → store

> **Keep documents small** — a few pages or one chapter (a few thousand words). Everything runs in Google's cloud, so your laptop's speed/RAM don't matter; this cap just keeps you comfortably inside the free daily quota.

In [ ]:
# Cell 2 — load your document
from pypdf import PdfReader
import urllib.request

FILENAME = "my_notes.pdf"         # ← your file (.pdf or .txt), uploaded via the folder icon
SAMPLE   = "sample-os-notes.txt"  # instructor's sample: OS lecture notes + an ILLUSTRATIVE page of academic regulations.
                                  # Used automatically if FILENAME is missing — fine for the lab, but your own notes make a better capstone.
SAMPLE_URL = "https://raw.githubusercontent.com/intrepidkarthi/generative-ai-projects-for-students/main/labs/session-4/sample-os-notes.txt"

if not os.path.exists(FILENAME):
    print(f"'{FILENAME}' not found — using the instructor sample '{SAMPLE}'. Upload your file and set FILENAME to switch.")
    if not os.path.exists(SAMPLE):
        try:
            urllib.request.urlretrieve(SAMPLE_URL, SAMPLE)
            print("downloaded", SAMPLE)
        except Exception as e:
            raise FileNotFoundError(f"could not download the sample ({type(e).__name__}). "
                                    f"Upload '{SAMPLE}' from the course site via the folder icon, then re-run this cell.") from e
    FILENAME = SAMPLE

if FILENAME.endswith(".pdf"):
    text = "\n".join(page.extract_text() or "" for page in PdfReader(FILENAME).pages)
else:
    text = open(FILENAME, encoding="utf-8").read()

print(len(text), "characters loaded from", FILENAME)
print(text[:400])   # sanity: is this YOUR text, readable?
if len(text.strip()) < 200:
    raise ValueError("Too little readable text was extracted. This may be a scanned/image-only PDF; switch to a text PDF or set FILENAME = SAMPLE.")

In [ ]:
# Cell 3 — chunk: paragraphs, merged to a target size, with overlap
def chunk_text(text, target=800, overlap=150):
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) > target and cur:
            chunks.append(cur.strip())
            cur = cur[-overlap:] + " " + p
        else:
            cur += " " + p
    if cur.strip(): chunks.append(cur.strip())
    return chunks

chunks = chunk_text(text)

# Keep it free-tier-friendly, but sample across the document rather than silently
# indexing only its opening pages. This is a teaching cap, not a coverage guarantee.
MAX_CHUNKS = 60
if len(chunks) > MAX_CHUNKS:
    original_count = len(chunks)
    selected = np.linspace(0, len(chunks) - 1, MAX_CHUNKS, dtype=int)
    chunks = [chunks[i] for i in selected]
    print(f"note: {original_count} chunks -> sampled {MAX_CHUNKS} across the document to stay within the teaching quota")

if not chunks:
    raise ValueError("No usable chunks were created. Check the document extraction before embedding.")

print(len(chunks), "chunks")
print("--- sample chunk ---\n", chunks[len(chunks)//2][:300])

In [ ]:
# Cell 4 — embed all chunks (batched), store as one numpy matrix
def embed(texts):
    """One 768-d vector per text.
    Why the wrapping: gemini-embedding-2 folds a bare list of strings into ONE aggregated embedding
    (60 chunks → 1 vector, silently). Wrapping each text as its own Content gives one vector per chunk."""
    if MOCK:   # offline: deterministic hashed bag-of-words (crc32, so it matches across sessions) — search still ranks by word overlap
        M = np.zeros((len(texts), 768))
        for i, t in enumerate(texts):
            for w in re.findall(r"[a-z0-9]{4,}", t.lower()):      # skip "the", "of", "is"…
                M[i, zlib.crc32(w.encode()) % 768] += 1
        return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)
    res = client.models.embed_content(
        model=EMBED_MODEL,
        contents=[types.Content(parts=[types.Part.from_text(text=t)]) for t in texts],
        config=types.EmbedContentConfig(output_dimensionality=768))
    assert len(res.embeddings) == len(texts), \
        f"expected {len(texts)} vectors, got {len(res.embeddings)} — each text must be wrapped as its own Content"
    return np.array([e.values for e in res.embeddings])

vecs = []
B = 20
for i in range(0, len(chunks), B):
    vecs.append(embed(chunks[i:i+B]))
    print(f"embedded {min(i+B, len(chunks))}/{len(chunks)}")
    time.sleep(1)   # be polite to the free tier

chunk_vecs = np.vstack(vecs)
chunk_vecs = chunk_vecs / np.linalg.norm(chunk_vecs, axis=1, keepdims=True)  # normalize once
print("vector store:", chunk_vecs.shape, "← this numpy array IS your vector database")
assert chunk_vecs.shape[0] == len(chunks), "one vector per chunk — if this fails, re-read embed()"

# capstone insurance — Labs 5 and 6 reload these from Drive instead of re-embedding
np.save(os.path.join(SAVE_DIR, "chunk_vecs.npy"), chunk_vecs)
json.dump(chunks, open(os.path.join(SAVE_DIR, "chunks.json"), "w", encoding="utf-8"))
print("saved chunk_vecs.npy + chunks.json →", os.path.abspath(SAVE_DIR))

### ✓ Checkpoint 1 — chunk count + vector store shape printed.

---
## Part B — Semantic search (the R in RAG)

In [ ]:
# Cell 5 — search = one matrix multiply
def search(query, k=3):
    qv = embed([query])[0]
    qv = qv / np.linalg.norm(qv)
    scores = chunk_vecs @ qv                  # cosine similarity, all chunks at once
    top = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in top]

# sanity check with 3 queries — these work on the sample document (see sample-queries.md)
for q in [                                   # ← replace with questions about YOUR document
    "What is the difference between a process and a thread?",
    "What are the four conditions for deadlock?",
    "What attendance percentage is required to write the end-semester exam?",
]:
    print("=" * 60, "\nQ:", q)
    for s, c in search(q):
        print(f"  {s:.2f} | {c[:110]}...")

### Do the top chunks LOOK right?
If not: chunks too big/small (tune `target`)? PDF extracted garbage (check Cell 2 output)? Query too vague?

### ✓ Checkpoint 2 — three sane searches.

---
## Part C — The full RAG loop (the A and G)

In [ ]:
# Cell 6 — grounded, cited answers
RAG_TEMPLATE = """Answer the question using ONLY the context below.
Cite which chunk you used, like [1] or [2].
If the answer is not in the context, reply exactly: "I don't know based on the provided documents."

CONTEXT:
{context}

QUESTION: {question}"""

def rag_ask(question, k=3, show_chunks=False):
    hits = search(question, k)
    context = "\n\n".join(f"[{i+1}] {c}" for i, (s, c) in enumerate(hits))
    if show_chunks:
        for i, (s, c) in enumerate(hits): print(f"  [{i+1}] ({s:.2f}) {c[:80]}...")
    return ask(RAG_TEMPLATE.format(context=context, question=question))

print(rag_ask("What is the minimum mark needed to pass a theory course?", show_chunks=True))   # ← replace with a question about YOUR document

In [ ]:
# Cell 7 — interrogate your own notes (5 real questions)
for q in [                                   # ← replace with questions about YOUR document
    "How does round-robin scheduling work, and what happens if the time quantum is very large?",
    "Compare paging and segmentation in one line each.",
    "What is Belady's anomaly and which page-replacement algorithm suffers from it?",
    "How is the internal assessment for a theory course split?",
    "Who is the Head of the CSE department at TCE?",   # NOT in the document — it must say "I don't know"
]:
    print("=" * 60, "\nQ:", q, "\n")
    print(rag_ask(q), "\n")

## Part D — Break it honestly

1. Ask something **definitely NOT in your documents** → does it say "I don't know"? (If it invents instead, strengthen the ONLY/escape-hatch lines — this is real prompt hardening.)
2. Ask something whose answer is **split across two places** → does it get half the truth?

### ✓ Checkpoint 3 — one honest failure + what you changed to fix (or why it's hard).

---
## Stretch goals

In [ ]:
# Stretch 1 — RAG eval (your S2 harness, now grading your app)
# this becomes 25% of your capstone score — aim for 10 examples, not 5
rag_tests = [                                # ← replace with YOUR document's questions + key facts
    {"q": "What is the minimum mark needed to pass a theory course?",          "expected": "45"},
    {"q": "What happens to round-robin if the time quantum is very large?",     "expected": "fcfs"},
    {"q": "Which page-replacement algorithm suffers from Belady's anomaly?",   "expected": "fifo"},
    {"q": "How many marks is the internal assessment of a theory course?",     "expected": "50"},
    {"q": "Who is the Head of the CSE department at TCE?",                     "expected": "don't know"},   # the refusal path
]
def norm(s): return re.sub(r"[^a-z0-9 ]", "", s.lower())
hits = 0
for t in rag_tests:
    ans = rag_ask(t["q"])
    ok = norm(t["expected"]) in norm(ans); hits += ok
    print("✓" if ok else "✗", t["q"])
print(f"RAG score: {hits}/{len(rag_tests)}")

In [ ]:
# Stretch 2 — does k matter?
q = "Summarise every rule about arrears and re-appearance."   # ← needs broad context; replace for YOUR document
for k in [1, 3, 5]:
    print(f"===== k={k} =====")
    print(rag_ask(q, k=k)[:300], "\n")
# Small k: may miss context. Big k: noise + tokens. Where's YOUR sweet spot?

### S3 · Rerank — retrieve wide, then narrow

The single biggest RAG upgrade after chunking, in about ten lines and no new library. Run it on a question your plain search gets *slightly* wrong — the interesting result is a chunk climbing from position 9 into the top 4.

### S4 · Two scores, never one



In [ ]:
# Stretch 3 — rerank: retrieve wide, then narrow (the biggest upgrade after chunking)
# Your search() is a BI-ENCODER: question and chunks were embedded separately, which is what
# makes it fast. It is good at RECALL (the right chunk is usually in the top 20) and mediocre
# at RANKING (it may sit at position 9 while you only take 3). A reranker reads the question
# and each chunk TOGETHER and re-orders them. Here it is, in one model call.
import json as _json

def rerank(query, candidates, keep=4):
    """candidates: list of (score, chunk). Returns the best `keep`, model-ordered."""
    listing = "\n\n".join(f"[{i}] {c[:400]}" for i, (s, c) in enumerate(candidates))
    prompt = f"""Rank these passages by how well they help answer the question.
Return ONLY the ids of the {keep} most useful, best first.

QUESTION: {query}

PASSAGES:
{listing}"""
    if MOCK:
        raw = ask(prompt).removeprefix("[MOCK] ")      # canned ids, so the cell runs offline
    else:
        raw = client.models.generate_content(
            model=MODEL, contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json",     # ← schema, not begging (Session 3)
                response_schema={"type":"object",
                                 "properties":{"ids":{"type":"array","items":{"type":"integer"}}},
                                 "required":["ids"]})).text or "{}"
    ids = _json.loads(raw).get("ids", [])[:keep]
    return [candidates[i] for i in ids if 0 <= i < len(candidates)]

q = "If I score well in internals but badly in the end-semester, do I pass?"   # ← replace with one YOUR plain search gets slightly wrong
wide = search(q, k=20)                  # cheap + approximate
best = rerank(q, wide, keep=4)          # expensive + accurate, on 20 items only

print("BEFORE (top 4 by embedding):")
for s, c in wide[:4]:  print(f"  {s:.2f} | {c[:90]}...")
print("\nAFTER (reranked):")
for s, c in best:      print(f"  {s:.2f} | {c[:90]}...")


In [ ]:
# Stretch 4 — measure retrieval and generation SEPARATELY (two scores, never one)
# Label which chunk SHOULD win for each question, then you can tell WHERE you are broken:
#   low recall@k        -> ingest/chunking is broken (no prompt can save you)
#   good recall, low MRR -> ranking is broken       -> add the reranker above
#   good retrieval, bad answers -> grounding/prompt is broken
probe = [                                    # ← replace with YOUR document (this takes 5 minutes and pays for itself immediately)
    # (question, a distinctive phrase that appears in the chunk that SHOULD be retrieved)
    ("What are the four conditions for deadlock?",                       "circular wait"),
    ("What attendance is needed to write the end-semester exam?",        "75% attendance"),
    ("What does a very large time quantum do to round robin?",           "degenerates to FCFS"),
    ("Which page-replacement algorithm suffers from Belady's anomaly?",  "Belady's anomaly"),
    ("How is the internal assessment split?",                            "scaled to 30"),
]
K = 5
recall = rr = 0
for q, needle in probe:
    hits = [c for s, c in search(q, k=K)]
    rank = next((i + 1 for i, c in enumerate(hits) if needle.lower() in c.lower()), None)
    recall += rank is not None
    rr     += 1 / rank if rank else 0
    print(f"{'✓' if rank else '✗'} rank={rank}  {q[:52]}")
n = len(probe)
print(f"\nrecall@{K} = {recall}/{n} = {recall/n:.0%}   <- your CEILING on answer accuracy")


## Capstone foundation — saved?

**File → Save** keeps your code — not your uploaded PDF or embeddings. Cell 4 wrote `chunk_vecs.npy` + `chunks.json` to your Drive (`MyDrive/genai`); Labs 5 and 6 have a reload cell that reads them back instead of re-embedding. This notebook returns in Sessions 5 and 6: it gains tools after lunch and gets attacked (then hardened) in the finale. Short break — then AI that *does things*.